# 🚀 Frontiir AI Chat — Google Colab Setup

> **မှတ်ချက်:** Runtime → Change runtime type → **T4 GPU** ရွေးပြီးမှ run ပါ

---
## Cell တွေကို အစဉ်တကျ Run ပါ (Shift+Enter)

## Step 1 — GPU စစ်ဆေး

In [ ]:
!nvidia-smi | grep -E 'Tesla|NVIDIA|Driver'
print('✅ GPU ready!' if __import__('subprocess').call(['nvidia-smi'], stdout=__import__('subprocess').DEVNULL, stderr=__import__('subprocess').DEVNULL) == 0 else '⚠️ No GPU — CPU mode')

## Step 2 — Go Install

In [ ]:
import os

# Install Go 1.22 official binary
!wget -q https://go.dev/dl/go1.22.4.linux-amd64.tar.gz
!tar -C /usr/local -xzf go1.22.4.linux-amd64.tar.gz
!rm go1.22.4.linux-amd64.tar.gz

# Set PATH
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']

# Verify
!go version

## Step 3 — Project Upload

Local PC မှာ zip လုပ်ပါ (terminal မှာ):
```bash
cd ~/FrontiirProjects/AI
zip -r AI_CHAT.zip AI_CHAT/ --exclude 'AI_CHAT/.env'
```
ပြီးရင် cell ကို run ပြီး zip ကို upload လုပ်ပါ

In [ ]:
from google.colab import files
print('📂 AI_CHAT.zip ကို choose လုပ်ပါ...')
uploaded = files.upload()
print('✅ Upload done:', list(uploaded.keys()))

## Step 4 — Unzip Project

In [ ]:
!unzip -q AI_CHAT.zip -d /content/
!echo '✅ Files:' && ls /content/AI_CHAT/

## Step 5 — .env File ဖန်တီး

> ⚠️ NGROK_TOKEN ကို ဒီ notebook မှာ save မလုပ်ပါနှင့်

In [ ]:
ngrok_token = input('ngrok token ထည့်ပါ: ')

env_content = f"""LLM_PROVIDER=ollama
OLLAMA_URL=http://localhost:11434
OLLAMA_MODEL=qwen2.5:7b
QDRANT_URL=http://localhost:6333
CPE_API_URL=http://localhost:9000/api/cpe
CUSTOMER_API_URL=http://localhost:9000/api/customer
NGROK_TOKEN={ngrok_token}
"""

with open('/content/AI_CHAT/.env', 'w') as f:
    f.write(env_content)

print('✅ .env created (token hidden)')

## Step 6 — Install GPU Tools + Ollama

In [ ]:
!apt-get install -y lshw pciutils -q
!curl -fsSL https://ollama.com/install.sh | sh
print('✅ Ollama installed')

## Step 7 — Ollama Server Start

In [ ]:
import subprocess, time, os

env = {**os.environ, 'OLLAMA_HOST': '0.0.0.0:11434'}
proc = subprocess.Popen(
    ['ollama', 'serve'],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
print(f'Ollama PID: {proc.pid}')
print('Waiting 40s for GPU init...')
time.sleep(40)

# Test
import urllib.request, json
try:
    with urllib.request.urlopen('http://localhost:11434/api/tags') as r:
        print('✅ Ollama is ready:', r.read().decode()[:50])
except Exception as e:
    print('❌ Ollama not ready:', e)

## Step 8 — Pull Models

> qwen2.5:7b = 4.7GB — ၁၀-၁၅ မိနစ် ကြာနိုင်

In [ ]:
import os
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']

print('📥 Pulling qwen2.5:7b (chat model)...')
!ollama pull qwen2.5:7b

print('\n📥 Pulling nomic-embed-text (embedding model)...')
!ollama pull nomic-embed-text

print('\n✅ Models ready!')
!ollama list

## Step 9 — ngrok Tunnel Start

In [ ]:
!pip install pyngrok -q

import os
from pyngrok import ngrok

# Read token from .env
with open('/content/AI_CHAT/.env') as f:
    for line in f:
        if line.startswith('NGROK_TOKEN='):
            token = line.strip().split('=', 1)[1]
            break

ngrok.set_auth_token(token)
tunnel = ngrok.connect(11434)
ollama_url = tunnel.public_url

# Update .env with real URL
with open('/content/AI_CHAT/.env') as f:
    content = f.read()
content = content.replace('OLLAMA_URL=http://localhost:11434', f'OLLAMA_URL={ollama_url}')
with open('/content/AI_CHAT/.env', 'w') as f:
    f.write(content)

print('✅ Ollama is public!')
print()
print('  OLLAMA_URL =', ollama_url)
print()
print('  Local .env မှာ ဒီ line ကို ထည့်ပါ:')
print(f'  OLLAMA_URL={ollama_url}')

## Step 10 — Ingest Training Data (Optional)

> Qdrant ကို local မှာ run ထားရမည်

In [ ]:
import os
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
os.chdir('/content/AI_CHAT')

# Test Ollama connection
!go run ./cmd/ingest-jsonl --file=train/train.jsonl
print('✅ train.jsonl ingested')

## ✅ Setup Complete!

**Local PC မှာ လုပ်ရမည်:**

1. `.env` ထဲ `OLLAMA_URL=https://xxxx.ngrok-free.app` ထည့်ပါ  
2. `go run .` run ပါ  
3. `http://localhost:8080` ဖွင့်ပါ

---
⚠️ **မှတ်ရန်:** Colab session 12 နာရီ ကုန်ရင် URL ပြောင်းမယ် → Step 9 ကို ပြန် run → URL အသစ် → local `.env` ပြင်ပါ

## 🔄 Session Reconnect (Session သေပြီး ပြန်ချိတ်ရင်)

Session expire ဖြစ်ရင် ဒီ cell တွေပဲ ပြန် run ပါ:

In [ ]:
# ── RECONNECT CELL (Session သေပြီး ပြန် start ရင် ဒီ cell ကိုပဲ run) ──
import subprocess, time, os
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']

# 1. Ollama ပြန် start
env = {**os.environ, 'OLLAMA_HOST': '0.0.0.0:11434'}
proc = subprocess.Popen(['ollama', 'serve'], env=env,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print(f'Ollama PID: {proc.pid} — waiting 40s...')
time.sleep(40)

# 2. ngrok ပြန် start
!pip install pyngrok -q
from pyngrok import ngrok

with open('/content/AI_CHAT/.env') as f:
    for line in f:
        if line.startswith('NGROK_TOKEN='):
            token = line.strip().split('=', 1)[1]
            break

ngrok.set_auth_token(token)
tunnel = ngrok.connect(11434)
print('✅ NEW OLLAMA_URL =', tunnel.public_url)
print()
print('Local .env မှာ ဒီ URL ကို update ပါ:')
print(f'OLLAMA_URL={tunnel.public_url}')